##Question 1 - 1D Convolution
In the lecture, you have learned convolution in its basic implementation and tiled optimization. This exercise will develop both implementations and compare their performance. You will leverage tiling and the shared memory in the optimized version to reduce global memory accesses. The input array N has (N) floats, where N is given by the user. The mask M has a fixed size of 5. The output array P also has (N) floats. You can assume out-of-boundary elements to have value 0. You can modify the code skeleton hereDownload code skeleton here.




In [ ]:
%%writefile convolution.cu
#include <cuda_runtime_api.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/time.h>

#define gpuCheck(stmt)                                               \
  do {                                                               \
      cudaError_t err = stmt;                                        \
      if (err != cudaSuccess) {                                      \
          printf("ERROR. Failed to run stmt %s\n", #stmt);           \
          break;                                                     \
      }                                                              \
  } while (0)

struct timeval t_start, t_end;
void cputimer_start(){
  gettimeofday(&t_start, 0);
}

void cputimer_stop(const char* info){
  gettimeofday(&t_end, 0);
  double time = (1000000.0*(t_end.tv_sec-t_start.tv_sec) + t_end.tv_usec-t_start.tv_usec);
  printf("Timing - %s. \t\tElasped %.0f microseconds \n", info, time);
}

#define MASK_WIDTH 5
#define TILE_WIDTH 512

#define RADIUS (MASK_WIDTH / 2)
#define OUTPUT_WIDTH (TILE_WIDTH - 2 * RADIUS)


__global__ void convolution_1D_basic(float *N, float *M, float *P, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {

        float sum = 0.0f;

        for (int j = -RADIUS; j <= RADIUS; j++) {

            int input_idx = idx + j;

            // Check boundaries
            if (input_idx >= 0 && input_idx < n) {
                sum += N[input_idx] * M[j + RADIUS];
            }
        }

        P[idx] = sum;
    }

}

__global__ void convolution_1D_tiled(float *N, float *M, float *P, int n)
{

  __shared__ float input_tile[TILE_WIDTH];

    int tx = threadIdx.x;

    // First output element belonging to this block
    int output_start = blockIdx.x * OUTPUT_WIDTH;

    // Global input index, including left halo
    int input_idx = output_start + tx - RADIUS;

    // Load input into shared memory
    if (input_idx >= 0 && input_idx < n) {
        input_tile[tx] = N[input_idx];
    }
    else {
        input_tile[tx] = 0.0f;
    }

    // Wait until all threads finish loading
    __syncthreads();

    // Only interior threads calculate output
    if (tx >= RADIUS && tx < TILE_WIDTH - RADIUS) {

        int output_idx = output_start + tx - RADIUS;

        if (output_idx < n) {

            float sum = 0.0f;

            for (int j = -RADIUS; j <= RADIUS; j++) {

                sum += input_tile[tx + j] * M[j + RADIUS];
            }

            P[output_idx] = sum;
        }
    }

}

int main(int argc, char *argv[]) {

  // Read the arguments from the command line
  int n = atoi(argv[1]);

  size_t input_size = n * sizeof(float);
  size_t mask_size = MASK_WIDTH * sizeof(float);

  float *hostN; // The input array N of length N
  float *hostM; // The 1D mask M of length MASK_WIDTH
  float *hostP; // The output array P of length N
  float *hostBasic;

  cputimer_start();
  hostN = (float*)malloc(input_size);
  hostM = (float*)malloc(mask_size);
  hostP = (float*)malloc(input_size);
  hostBasic = (float*)malloc(input_size);

  if ( !hostN || !hostM || !hostP || !hostBasic) {
    fprintf(stderr, "Host memory allocation failed.\n");
    return 1;
  }
  cputimer_stop("Allocated host memory");


  float *deviceN;
  float *deviceM;
  float *deviceP;

  cputimer_start();
  gpuCheck(cudaMalloc((void**)&deviceN, input_size));
  gpuCheck(cudaMalloc((void**)&deviceM, mask_size));
  gpuCheck(cudaMalloc((void**)&deviceP, input_size));
  cputimer_stop("Allocated device memory");


  cputimer_start();
  srand(1234);

  for (int i = 0; i < n; i++) {
      hostN[i] = (float)rand() / RAND_MAX;
      hostP[i] = 0.0f;
      hostBasic[i] = 0.0f;
  }

  // Fixed convolution mask
  hostM[0] = -0.25f;
  hostM[1] =  0.5f;
  hostM[2] =  1.0f;
  hostM[3] =  0.5f;
  hostM[4] =  0.25f;
  cputimer_stop("Initialized input and mask");


  cputimer_start();
  gpuCheck(cudaMemcpy(deviceN, hostN, input_size, cudaMemcpyHostToDevice));
  gpuCheck(cudaMemcpy(deviceM, hostM, mask_size, cudaMemcpyHostToDevice));
  cputimer_stop("Copying data to the GPU");


  /* Call the basic kernel */
  int blockSize = 256;
  int gridSize = (n + blockSize - 1) / blockSize;

  cputimer_start();

  convolution_1D_basic<<<gridSize, blockSize>>>(deviceN, deviceM, deviceP, n);
  gpuCheck(cudaGetLastError());
  gpuCheck(cudaDeviceSynchronize());
  cputimer_stop("Finished 1D convolution(basic)");

  cputimer_start();
  gpuCheck(cudaMemcpy(hostBasic, deviceP, input_size, cudaMemcpyDeviceToHost));
  cputimer_stop("Copying output P to the CPU and print out the results");

  /* Call the tiled kernel */
  int tiledGridSize = (n + OUTPUT_WIDTH - 1) / OUTPUT_WIDTH;

  cputimer_start();
  convolution_1D_tiled<<<tiledGridSize, TILE_WIDTH>>>(deviceN, deviceM, deviceP, n);

  gpuCheck(cudaGetLastError());
  gpuCheck(cudaDeviceSynchronize());
  cputimer_stop("Finished 1D convolution(tiled)");

  cputimer_start();
  gpuCheck(cudaMemcpy(hostP, deviceP, input_size, cudaMemcpyDeviceToHost));
  cputimer_stop("Copying output P to the CPU and print out the results");

  // --------------------------------------------------
  // Validate results
  // --------------------------------------------------
  printf("\n--- Validation ---\n");

    bool correct = true;

    float max_error = 0.0f;

    for (int i = 0; i < n; i++) {

        // CPU reference calculation
        float reference = 0.0f;

        for (int j = -RADIUS; j <= RADIUS; j++) {

            int input_idx = i + j;

            if (input_idx >= 0 && input_idx < n) {
                reference +=
                    hostN[input_idx] * hostM[j + RADIUS];
            }
        }

        float error_basic = fabsf(hostBasic[i] - reference);
        float error_tiled = fabsf(hostP[i] - reference);

        if (error_basic > max_error)
            max_error = error_basic;

        if (error_tiled > max_error)
            max_error = error_tiled;

        float tolerance = 1e-5f * (1.0f + fabsf(reference));

        if (error_basic > tolerance ||
            error_tiled > tolerance) {

            correct = false;

            if (i < 10) {
                printf("Mismatch at index %d:\n", i);
                printf("  CPU reference: %f\n", reference);
                printf("  Basic GPU:     %f\n", hostBasic[i]);
                printf("  Tiled GPU:     %f\n", hostP[i]);
            }
        }
    }

    if (correct) {
        printf("SUCCESS: Both GPU implementations match "
               "the CPU reference!\n");
    }
    else {
        printf("ERROR: Validation failed!\n");
    }

    printf("Maximum absolute error: %e\n", max_error);

    // --------------------------------------------------
    // Free memory
    // --------------------------------------------------

    cputimer_start();

    free(hostN);
    free(hostM);
    free(hostP);
    free(hostBasic);

    gpuCheck(cudaFree(deviceN));
    gpuCheck(cudaFree(deviceM));
    gpuCheck(cudaFree(deviceP));

    cputimer_stop("Free memory resources");

    return correct ? 0 : 1;
}


Overwriting convolution.cu


In [ ]:
!nvcc -O3 convolution.cu -o convolution

In [ ]:
!./convolution 1000

Timing - Allocated host memory. 		Elasped 5 microseconds 
Timing - Allocated device memory. 		Elasped 192582 microseconds 
Timing - Initialized input and mask. 		Elasped 33 microseconds 
Timing - Copying data to the GPU. 		Elasped 43 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 128 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 19 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 29 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 13 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00

--- First output elements ---
P[0] = 0.443406
P[1] = 0.675289
P[2] = 0.888827
P[3] = 1.065327
P[4] = 1.059407
P[5] = 0.881190
P[6] = 0.824811
P[7] = 0.869318
P[8] = 1.369871
P[9] = 1.569337
Timing - Free memory resources. 		Elasped 114 microseconds 


In [ ]:
!nvidia-smi

Tue Nov 17 15:01:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile query_gpu.cu
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int device;
    cudaDeviceProp prop;

    cudaError_t err = cudaGetDevice(&device);
    if (err != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    cudaGetDeviceProperties(&prop, device);

    printf("GPU: %s\n", prop.name);
    printf("Compute capability: %d.%d\n", prop.major, prop.minor);
    printf("Maximum threads per block: %d\n", prop.maxThreadsPerBlock);
    printf("Shared memory per block: %zu bytes\n", prop.sharedMemPerBlock);
    printf("Shared memory per multiprocessor: %zu bytes\n",
           prop.sharedMemPerMultiprocessor);
    printf("Number of multiprocessors: %d\n", prop.multiProcessorCount);

    return 0;
}

Writing query_gpu.cu


In [ ]:
!nvcc query_gpu.cu -o query_gpu
!./query_gpu

GPU: Tesla T4
Compute capability: 7.5
Maximum threads per block: 1024
Shared memory per block: 49152 bytes
Shared memory per multiprocessor: 65536 bytes
Number of multiprocessors: 40


In [ ]:
!./convolution 1024

Timing - Allocated host memory. 		Elasped 6 microseconds 
Timing - Allocated device memory. 		Elasped 204914 microseconds 
Timing - Initialized input and mask. 		Elasped 28 microseconds 
Timing - Copying data to the GPU. 		Elasped 41 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 139 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 21 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 30 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 13 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 129 microseconds 


In [ ]:
!./convolution 2048

Timing - Allocated host memory. 		Elasped 4 microseconds 
Timing - Allocated device memory. 		Elasped 186065 microseconds 
Timing - Initialized input and mask. 		Elasped 56 microseconds 
Timing - Copying data to the GPU. 		Elasped 68 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 135 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 20 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 29 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 14 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 117 microseconds 


In [ ]:
!./convolution 4096

Timing - Allocated host memory. 		Elasped 13 microseconds 
Timing - Allocated device memory. 		Elasped 179687 microseconds 
Timing - Initialized input and mask. 		Elasped 103 microseconds 
Timing - Copying data to the GPU. 		Elasped 40 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 184 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 24 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 31 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 17 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 116 microseconds 


In [ ]:
!./convolution 8192

Timing - Allocated host memory. 		Elasped 11 microseconds 
Timing - Allocated device memory. 		Elasped 199657 microseconds 
Timing - Initialized input and mask. 		Elasped 209 microseconds 
Timing - Copying data to the GPU. 		Elasped 47 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 141 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 28 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 31 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 21 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 135 microseconds 


In [ ]:
!./convolution 16384

Timing - Allocated host memory. 		Elasped 11 microseconds 
Timing - Allocated device memory. 		Elasped 162326 microseconds 
Timing - Initialized input and mask. 		Elasped 377 microseconds 
Timing - Copying data to the GPU. 		Elasped 67 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 127 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 34 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 29 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 28 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 127 microseconds 


In [ ]:
!nvcc -O3 -lineinfo convolution.cu -o convolution

In [ ]:
!nsys profile --trace=cuda --stats=true -o tiled_profile ./convolution 16384

Timing - Allocated host memory. 		Elasped 14 microseconds 
Timing - Allocated device memory. 		Elasped 653211 microseconds 
Timing - Initialized input and mask. 		Elasped 407 microseconds 
Timing - Copying data to the GPU. 		Elasped 77 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 1013 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 51 microseconds 
Timing - Finished 1D convolution(tiled). 		Elasped 62 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 34 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 233 microseconds 
Generating '/tmp/nsys-report-2a82.qdstrm'
[1/6] [========================100%] tiled_profile.nsys-rep
[2/6] [========================100%] tiled_profile.sqlite
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)

In [ ]:
!ncu --set basic --kernel-name regex:convolution_1D_tiled ./convolution 16384

Timing - Allocated host memory. 		Elasped 8 microseconds 
==PROF== Connected to process 12170 (/content/convolution)
Timing - Allocated device memory. 		Elasped 672542 microseconds 
Timing - Initialized input and mask. 		Elasped 502 microseconds 
Timing - Copying data to the GPU. 		Elasped 70 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 2827 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 69 microseconds 
==PROF== Profiling "convolution_1D_tiled": 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(tiled). 		Elasped 2484807 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 65 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 252 microseconds 
==PROF== Disconnected from process 12170
[12170] convolution@127.0.0.1
  convolution_1D_tiled(float *, float *, float *

In [ ]:
!ncu --set full --kernel-name regex:convolution_1D_tiled --export tiled_512_full ./convolution 16384

Timing - Allocated host memory. 		Elasped 6 microseconds 
==PROF== Connected to process 20144 (/content/convolution)
Timing - Allocated device memory. 		Elasped 561409 microseconds 
Timing - Initialized input and mask. 		Elasped 505 microseconds 
Timing - Copying data to the GPU. 		Elasped 80 microseconds 
Timing - Finished 1D convolution(basic). 		Elasped 1040 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 43 microseconds 
==PROF== Profiling "convolution_1D_tiled": 0%....50%....100% - 33 passes
Timing - Finished 1D convolution(tiled). 		Elasped 6900515 microseconds 
Timing - Copying output P to the CPU and print out the results. 		Elasped 73 microseconds 

--- Validation ---
SUCCESS: Both GPU implementations match the CPU reference!
Maximum absolute error: 0.000000e+00
Timing - Free memory resources. 		Elasped 481 microseconds 
==PROF== Disconnected from process 20144
==PROF== Report: /content/tiled_512_full.ncu-rep
